# mixup 复现（Colab 版）：CIFAR-10 + PreAct ResNet-18

**先做一件事**：点上方菜单「代码执行程序 / 运行时」→「更改运行时类型」→ 选 **T4 GPU** → 保存。不选 GPU 会慢到跑不完。

**这个笔记本干什么**：复现 mixup 论文 Figure 3a 的 CIFAR-10 实验。跑两组完全相同的训练，只差一个开关：
- 对照组（ERM，传统训练）：`alpha=0`
- 实验组（mixup）：`alpha=1.0`（论文 CIFAR 实验的原值）

**怎么用**：从上往下，每个代码格子左上角有个 ▶ 按钮，点一下等它跑完，再点下一个。

**时间预估（估计值）**：T4 上约 60~90 秒 1 个 epoch。冒烟测试 2 个 epoch 几分钟；正式实验每组 100 epoch 约 1.5~2.5 小时。两组可以分两次打开跑，每跑完一组先翻到最后把结果下载下来，防止会话断线丢数据。

超参数照抄论文 Section 3.2：200 epoch 的原计划按前辈建议缩短为 100 epoch，学习率衰减点等比例改为第 50、75 epoch。另加了一个小改动：前 5 个 epoch 做 lr warmup（学习率从 0.02 线性升到 0.1）——实测 lr=0.1 冷启动会在开头几十步发散（loss 变 nan），warmup 是标准解法，两组同等处理、不影响对照，report 里如实写明即可。

In [ ]:
# 第 1 格：检查 GPU（必须先选好 T4 GPU 再跑这格）
!nvidia-smi
import torch
print('PyTorch 版本：', torch.__version__)
print('GPU 可用：', torch.cuda.is_available())
if torch.cuda.is_available():
    print('显卡：', torch.cuda.get_device_name(0))
else:
    print('⚠️ 没检测到 GPU！先去 代码执行程序 → 更改运行时类型 → 选 T4 GPU，然后从这一格重新跑。')

## 第 2 格：全部代码（模型 + 数据 + mixup + 训练函数）

点运行就行，不用改任何东西。代码和本地版 `train.py` 是同一套，注释都保留着。

In [ ]:
# 第 2 格：导入 + 随机种子 + 模型定义（PreAct ResNet-18，CIFAR 版本）
import csv, json, os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False))

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        return self.linear(out)

def PreActResNet18():
    return PreActResNet(PreActBlock, [2, 2, 2, 2])

print('模型定义好了 ✅')

In [ ]:
# 第 3 格：数据 + mixup + 训练/测试 + 一键实验函数
def get_dataloaders(data_dir, batch_size, num_workers=2):
    # 论文引用的标准 CIFAR 增强：随机裁剪（padding 4）+ 随机水平翻转 + 通道归一化
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    trainset = torchvision.datasets.CIFAR10(root=data_dir, train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform_test)
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True,
                             num_workers=num_workers, pin_memory=True)
    testloader = DataLoader(testset, batch_size=256, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    return trainloader, testloader

def mixup_data(x, y, alpha):
    # 论文 Figure 1a 的做法：一个 batch 内随机打乱后两两混合
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)

def train_one_epoch(model, loader, criterion, optimizer, device, alpha):
    model.train()
    total_loss, n = 0.0, 0
    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if alpha > 0:  # 实验组：mixup
            inputs, t_a, t_b, lam = mixup_data(inputs, targets, alpha)
            loss = mixup_criterion(criterion, model(inputs), t_a, t_b, lam)
        else:          # 对照组：普通 ERM 训练
            loss = criterion(model(inputs), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        n += inputs.size(0)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        outputs = model(inputs)
        total_loss += criterion(outputs, targets).item() * inputs.size(0)
        correct += outputs.argmax(1).eq(targets).sum().item()
        n += inputs.size(0)
    return total_loss / n, correct / n

def run_experiment(alpha, epochs, milestones, seed, save_dir, warmup_epochs=5):
    # 一组完整实验：alpha=0 是对照组（ERM），alpha=1.0 是实验组（mixup）
    # 带 lr warmup：lr=0.1 冷启动会在前几十步发散（实测 ERM 第 42 步 nan），
    # 前 warmup_epochs 个 epoch 把 lr 从小线性升到 0.1 后即稳定。两组同等处理，对照公平。
    seed_everything(seed)
    os.makedirs(save_dir, exist_ok=True)
    config = dict(alpha=alpha, epochs=epochs, milestones=milestones, seed=seed,
                  batch_size=128, lr=0.1, wd=1e-4, warmup_epochs=warmup_epochs)
    with open(os.path.join(save_dir, 'args.json'), 'w') as f:
        json.dump(config, f, indent=2)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'设备：{device} | alpha={alpha} | epochs={epochs} | 结果存到 {save_dir}')
    trainloader, testloader = get_dataloaders('./data', 128)
    model = PreActResNet18().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    log_path = os.path.join(save_dir, 'log.csv')
    with open(log_path, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'test_loss', 'test_acc', 'lr'])
    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        # 本 epoch 学习率：warmup 阶段线性上升，之后在 milestones 处除以 10（论文规则）
        lr_now = 0.1
        if epoch <= warmup_epochs:
            lr_now = 0.1 * epoch / warmup_epochs
        else:
            for m in milestones:
                if epoch > m:
                    lr_now *= 0.1
        for g in optimizer.param_groups:
            g['lr'] = lr_now
        t0 = time.time()
        tr_loss = train_one_epoch(model, trainloader, criterion, optimizer, device, alpha)
        te_loss, te_acc = evaluate(model, testloader, criterion, device)
        with open(log_path, 'a', newline='') as f:
            csv.writer(f).writerow([epoch, f'{tr_loss:.4f}', f'{te_loss:.4f}', f'{te_acc:.4f}', f'{lr_now:.6f}'])
        if te_acc > best_acc:
            best_acc = te_acc
            torch.save(model.state_dict(), os.path.join(save_dir, 'best.pth'))
        print(f'[{epoch}/{epochs}] train_loss={tr_loss:.4f} test_loss={te_loss:.4f} '
              f'test_acc={te_acc * 100:.2f}% lr={lr_now}（{time.time() - t0:.0f}秒）')
    torch.save(model.state_dict(), os.path.join(save_dir, 'last.pth'))
    print(f'✅ 跑完了！最好 test_acc={best_acc * 100:.2f}%，日志在 {log_path}')
    return best_acc

print('训练函数准备好了 ✅')

## 第 4 格：冒烟测试（先跑这个！）

只跑 2 个 epoch，几分钟。目的是确认整条链路没问题：数据能下载、loss 在下降、不报错。数字不用看。

In [ ]:
# 第 4 格：冒烟测试（2 个 epoch）
run_experiment(alpha=1.0, epochs=2, milestones=[1], seed=0, save_dir='runs/smoke')

## 第 5、6 格：正式实验（每组约 1.5~2.5 小时，估计值）

一组跑完后，建议先跳到最后下载一次结果，再跑下一组，防止会话超时断线。

In [ ]:
# 第 5 格：对照组 ERM（传统训练，alpha=0）
run_experiment(alpha=0, epochs=100, milestones=[50, 75], seed=0, save_dir='runs/erm_s0')

In [ ]:
# 第 6 格：实验组 mixup（alpha=1.0，论文原值）
run_experiment(alpha=1.0, epochs=100, milestones=[50, 75], seed=0, save_dir='runs/mixup_s0')

## 第 7 格：画对比图 + 和论文对数字

两组都跑完后运行这格。论文的数字是 200 epoch 的（ERM 94.4% / mixup 95.8%），我们只跑 100 epoch，低一点正常，重点看 mixup 是否高于 ERM。

In [ ]:
# 第 7 格：画 ERM vs mixup 对比曲线，并打印和论文的对比
import matplotlib.pyplot as plt

def read_log(run_dir):
    epochs, accs = [], []
    with open(os.path.join(run_dir, 'log.csv')) as f:
        for row in csv.DictReader(f):
            epochs.append(int(row['epoch']))
            accs.append(float(row['test_acc']) * 100)
    return epochs, accs

plt.figure(figsize=(8, 5))
results = {}
for name, d in [('ERM 对照组', 'runs/erm_s0'), ('mixup 实验组', 'runs/mixup_s0')]:
    if os.path.exists(os.path.join(d, 'log.csv')):
        ep, acc = read_log(d)
        plt.plot(ep, acc, label=name)
        results[name] = max(acc)
        print(f'{name}：最好 test_acc = {max(acc):.2f}%（最后一个 epoch：{acc[-1]:.2f}%）')
    else:
        print(f'{name}：还没跑完（找不到 {d}/log.csv）')
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('ERM vs mixup on CIFAR-10 (PreAct ResNet-18, 100 epochs)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('compare.png', dpi=150)
plt.show()
print('对比图已存为 compare.png')
print()
print('参照（论文 200 epoch）：ERM 94.4% / mixup 95.8%')
if len(results) == 2:
    gap = results['mixup 实验组'] - results['ERM 对照组']
    print(f'我们的差距：mixup 比 ERM 高 {gap:.2f} 个百分点（论文是高 1.4 个）')

## 第 8 格：下载全部结果（写 report 要用）

把 `runs` 文件夹（log.csv、best.pth、args.json）和对比图打包下载。Colab 会话关掉后云端文件就没了，**务必下载**。

In [ ]:
# 第 8 格：打包下载结果
import shutil
from google.colab import files
shutil.make_archive('mixup_results', 'zip', '.', 'runs')
files.download('mixup_results.zip')
files.download('compare.png')
print('下载完成 ✅ 把 mixup_results.zip 和 compare.png 保存好，写 report 时要用')